# DL Polarity — Y2 ≥18-station events

Runs the PolarPicker Keras model directly on all waveforms in `waves_Y2_ge18.h5`.

## Model
- File: `FM6_RealTime/02-data/PolarPicker_unified_TMSF_001.keras`
- Input: `(N, 64, 1)` — 64 samples at 100 Hz (resampled from 200 Hz), window [-0.32, +0.31]s
- Output 1: `(N, 64, 1)` — waveform reconstruction (autoencoder, not used here)
- Output 2: `(N, 2)` — softmax `[P(Down), P(Up)]`
- Polarity: `argmax(output2)` → 0=Down (-1), 1=Up (+1)
- Confidence: `max(output2)` → 0.5–1.0

## Output
- `02-data/E_Po/dl_polarity_Y2_ge18.h5`
  - `/polarity/<STA>`   shape (N,)  int8   — +1=Up, -1=Down, 0=no waveform
  - `/confidence/<STA>` shape (N,)  float32 — 0.5–1.0 (NaN where no waveform)
  - `/event_ids`        shape (N,)  int64

## Environment
Run with: `/opt/miniconda3/envs/FM_RT/bin/jupyter nbconvert --to notebook --execute dl_polarity_Y2_ge18.ipynb`
or open in Jupyter with FM_RT kernel.

In [1]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import h5py
import keras
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('/Users/mczhang/Documents/GitHub/FM7/01-scripts/python')))
from pipeline_config import (
    STATIONS_ALL, EMAIL_PLOTS_DIR, DL_CONF_MIN,
    PROJECT, DATA
)

# Paths
WAVES_H5   = DATA / 'A_ID' / 'waves_Y2_ge18.h5'
OUT_H5     = DATA / 'E_Po' / 'dl_polarity_Y2_ge18.h5'
DL_MODEL   = Path('/Users/mczhang/Documents/GitHub/FM6_RealTime/02-data/PolarPicker_unified_TMSF_001.keras')
OUT_H5.parent.mkdir(parents=True, exist_ok=True)
EMAIL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Y2 station names (B-suffix for 2F main array)
STATIONS_Y2 = ['AS1','AS2','CC1','EC1','EC2','EC3','ID1',
                '01B','02B','03B','04B','05B','06B','07B',
                '08B','09B','10B','11B','12B','13B','14B','15A']

DL_FS     = 100    # model input sample rate
DL_N      = 64     # model input length
SRC_FS    = 200    # waveform sample rate in h5
BATCH     = 512    # prediction batch size

print(f'keras {keras.__version__}')
print(f'Waves  : {WAVES_H5}')
print(f'Model  : {DL_MODEL}')
print(f'Output : {OUT_H5}')

2026-03-25 15:06:55.694378: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


keras 3.12.1
Waves  : /Users/mczhang/Documents/GitHub/FM7/02-data/A_ID/waves_Y2_ge18.h5
Model  : /Users/mczhang/Documents/GitHub/FM6_RealTime/02-data/PolarPicker_unified_TMSF_001.keras
Output : /Users/mczhang/Documents/GitHub/FM7/02-data/E_Po/dl_polarity_Y2_ge18.h5


In [2]:
# ── Load model ────────────────────────────────────────────────────────────────
print('Loading model...')
t0 = time.time()
model = keras.models.load_model(str(DL_MODEL))
print(f'  Loaded in {time.time()-t0:.1f}s')
print(f'  Input  shape : {model.input_shape}')
print(f'  Output shapes: {[o.shape for o in model.outputs]}')
print(f'  Parameters   : {model.count_params():,}')

Loading model...


  Loaded in 0.2s
  Input  shape : (None, 64, 1)
  Output shapes: [(None, 64, 1), (None, 2)]
  Parameters   : 17,043


In [3]:
# ── Load waveforms ────────────────────────────────────────────────────────────
print('Loading waveforms...')
with h5py.File(WAVES_H5, 'r') as hf:
    event_ids = hf['event_ids'][:]
    waves = {}   # sta -> (N, 264) float32
    for sta in STATIONS_Y2:
        key = f'waveforms/{sta}'
        if key in hf:
            waves[sta] = hf[key][:]

N = len(event_ids)
print(f'  Events: {N:,}')
print(f'  Stations with waveforms: {len(waves)}')
for sta in STATIONS_Y2:
    if sta in waves:
        ok = int(np.sum(~np.isnan(waves[sta][:, 0])))
        print(f'    {sta}: {ok:>5,}/{N}  ({ok/N*100:.0f}%)')

Loading waveforms...
  Events: 1,182
  Stations with waveforms: 22
    AS1: 1,178/1182  (100%)
    AS2: 1,179/1182  (100%)
    CC1: 1,181/1182  (100%)
    EC1: 1,181/1182  (100%)
    EC2: 1,173/1182  (99%)
    EC3: 1,040/1182  (88%)
    ID1: 1,069/1182  (90%)
    01B:   735/1182  (62%)
    02B: 1,058/1182  (90%)
    03B:   862/1182  (73%)
    04B:   892/1182  (75%)
    05B:   628/1182  (53%)
    06B: 1,126/1182  (95%)
    07B: 1,111/1182  (94%)
    08B: 1,094/1182  (93%)
    09B: 1,110/1182  (94%)
    10B: 1,125/1182  (95%)
    11B: 1,052/1182  (89%)
    12B: 1,113/1182  (94%)
    13B:   880/1182  (74%)
    14B:   971/1182  (82%)
    15A:     0/1182  (0%)


In [4]:
def prepare_input(wave_264):
    """
    Convert a (N, 264) float32 array at 200 Hz to (N, 64, 1) at 100 Hz.

    Steps:
    1. Downsample 200→100 Hz: take every other sample → (N, 132)
    2. Slice first 64 samples → (N, 64)  = [-0.32, +0.31]s
    3. Normalize each waveform to unit max-abs amplitude
    4. Reshape to (N, 64, 1) for Conv1D input

    Rows with all-NaN (no waveform) get a zero vector.
    Returns: X (N, 64, 1) float32, valid_mask (N,) bool
    """
    N = len(wave_264)
    valid = ~np.isnan(wave_264[:, 0])   # True where waveform exists

    # Downsample: every other sample
    w100 = wave_264[:, ::2]             # (N, 132)
    w64  = w100[:, :DL_N].copy()        # (N, 64)

    # Normalize to unit max-abs (avoid divide-by-zero)
    peak = np.max(np.abs(w64), axis=1, keepdims=True)
    peak[peak == 0] = 1.0
    w64 = w64 / peak

    # Zero out invalid rows
    w64[~valid] = 0.0

    return w64[:, :, np.newaxis].astype(np.float32), valid


print('prepare_input() defined.')
# Quick sanity check on one station
X_test, vm = prepare_input(waves['AS1'])
print(f'  Test input shape: {X_test.shape}  valid: {vm.sum()}/{len(vm)}')

prepare_input() defined.
  Test input shape: (1182, 64, 1)  valid: 1178/1182


In [5]:
# ── Run DL predictions ────────────────────────────────────────────────────────
print(f'Running DL predictions (batch={BATCH})...')
t0 = time.time()

pol_out  = {}   # sta -> (N,) int8    +1/−1/0
conf_out = {}   # sta -> (N,) float32 0.5–1.0 or NaN

for sta in STATIONS_Y2:
    if sta not in waves:
        pol_out[sta]  = np.zeros(N, dtype=np.int8)
        conf_out[sta] = np.full(N, np.nan, dtype=np.float32)
        continue

    X, valid = prepare_input(waves[sta])

    # Predict — model returns [reconstruction, classification]
    _, class_probs = model.predict(X, batch_size=BATCH, verbose=0)
    # class_probs shape: (N, 2)  columns: [P(Down), P(Up)]

    polarity   = np.where(np.argmax(class_probs, axis=1) == 1, 1, -1).astype(np.int8)
    confidence = np.max(class_probs, axis=1).astype(np.float32)

    # Zero out invalid (no waveform) rows
    polarity[~valid]   = 0
    confidence[~valid] = np.nan

    pol_out[sta]  = polarity
    conf_out[sta] = confidence

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')

# Summary stats
total_picks  = sum((pol_out[s] != 0).sum() for s in STATIONS_Y2)
high_conf    = sum(((pol_out[s] != 0) & (conf_out[s] >= DL_CONF_MIN)).sum() for s in STATIONS_Y2)
print(f'Total picks:      {total_picks:,}')
print(f'High conf (≥{DL_CONF_MIN}): {high_conf:,}  ({high_conf/max(total_picks,1)*100:.1f}%)')

Running DL predictions (batch=512)...


Done in 2.1s
Total picks:      21,758
High conf (≥0.6): 20,363  (93.6%)


In [6]:
# ── Save to HDF5 ──────────────────────────────────────────────────────────────
print(f'Saving → {OUT_H5}')
with h5py.File(OUT_H5, 'w') as hf:
    hf.create_dataset('event_ids', data=event_ids)
    for sta in STATIONS_Y2:
        hf.create_dataset(f'polarity/{sta}',   data=pol_out[sta])
        hf.create_dataset(f'confidence/{sta}', data=conf_out[sta])
print(f'Saved. Size: {OUT_H5.stat().st_size/1e6:.1f} MB')

Saving → /Users/mczhang/Documents/GitHub/FM7/02-data/E_Po/dl_polarity_Y2_ge18.h5
Saved. Size: 0.2 MB


In [7]:
# ── Plots ─────────────────────────────────────────────────────────────────────
saved_plots = []

def savefig(fig, name):
    out = EMAIL_PLOTS_DIR / f'{name}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved → {out}')
    return str(out)

# ── Plot 1: Confidence histogram per station ──────────────────────────────────
fig, axes = plt.subplots(4, 6, figsize=(18, 12), sharey=True)
fig.suptitle('DL Confidence Distribution per Station — Y2 ≥18 sta', fontsize=13)
axes = axes.flatten()

for i, sta in enumerate(STATIONS_Y2):
    ax = axes[i]
    conf = conf_out[sta]
    valid_conf = conf[~np.isnan(conf)]
    if len(valid_conf) > 0:
        ax.hist(valid_conf, bins=20, range=(0.5, 1.0),
                color='steelblue', alpha=0.8, edgecolor='none')
        ax.axvline(DL_CONF_MIN, color='red', lw=1.5, ls='--')
        pct_hi = (valid_conf >= DL_CONF_MIN).mean() * 100
        ax.set_title(f'{sta}  {pct_hi:.0f}%≥{DL_CONF_MIN}', fontsize=8)
    else:
        ax.set_title(f'{sta}  (no data)', fontsize=8, color='gray')
    ax.set_xlim(0.5, 1.0)
    ax.tick_params(labelsize=6)

# hide unused subplots
for j in range(len(STATIONS_Y2), len(axes)):
    axes[j].set_visible(False)

fig.tight_layout()
saved_plots.append(savefig(fig, 'step06_confidence_hist'))

# ── Plot 2: Per-station summary bar chart ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pick_counts = [(pol_out[s] != 0).sum() for s in STATIONS_Y2]
hi_counts   = [((pol_out[s] != 0) & (conf_out[s] >= DL_CONF_MIN)).sum() for s in STATIONS_Y2]
up_counts   = [(pol_out[s] == 1).sum() for s in STATIONS_Y2]
dn_counts   = [(pol_out[s] == -1).sum() for s in STATIONS_Y2]

x = np.arange(len(STATIONS_Y2))
ax = axes[0]
ax.bar(x, pick_counts, color='steelblue', alpha=0.8, label='All picks')
ax.bar(x, hi_counts,   color='darkblue',  alpha=0.9, label=f'Conf ≥{DL_CONF_MIN}')
ax.set_xticks(x); ax.set_xticklabels(STATIONS_Y2, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Count'); ax.set_title('Pick count per station')
ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
ax.bar(x,  up_counts, color='tomato',    alpha=0.8, label='Up (+1)')
ax.bar(x, [-d for d in dn_counts], color='steelblue', alpha=0.8, label='Down (−1)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(STATIONS_Y2, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Count (positive=Up, negative=Down)')
ax.set_title('Up vs Down polarity per station')
ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
saved_plots.append(savefig(fig, 'step06_polarity_summary'))

# ── Plot 3: Sample waveforms with DL label (5 random events × 7 stations) ────
rng = np.random.default_rng(42)
# Pick 5 events where most stations have valid picks
n_valid_per_event = np.array([
    sum(1 for s in STATIONS_Y2 if pol_out[s][i] != 0)
    for i in range(N)
])
best_events = np.argsort(n_valid_per_event)[-20:]
sample_idx  = rng.choice(best_events, size=min(5, len(best_events)), replace=False)
show_stas   = ['AS1','CC1','EC1','01B','06B','09B','12B']

fig, axes = plt.subplots(len(show_stas), len(sample_idx),
                          figsize=(3*len(sample_idx), 2*len(show_stas)))
fig.suptitle('Sample waveforms with DL polarity label', fontsize=11)
t_axis = np.linspace(-0.32, 1.0, 264)

for col, ev_i in enumerate(sample_idx):
    eid = event_ids[ev_i]
    for row, sta in enumerate(show_stas):
        ax = axes[row, col]
        if sta in waves:
            w = waves[sta][ev_i]
            pol  = pol_out[sta][ev_i]
            conf = conf_out[sta][ev_i]
            if not np.isnan(w[0]):
                color = 'tomato' if pol == 1 else 'steelblue' if pol == -1 else 'gray'
                label = f"{'+' if pol==1 else '-' if pol==-1 else '?'} {conf:.2f}"
                ax.plot(t_axis, w, lw=0.6, color=color)
                ax.axvline(0, color='k', lw=0.8, ls='--')
                ax.set_title(label, fontsize=7, color=color)
            else:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=7, color='gray')
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(sta, fontsize=8)
        if row == 0:
            ax.set_xlabel(f'ev {eid}', fontsize=7)

fig.tight_layout()
saved_plots.append(savefig(fig, 'step06_sample_waveforms'))
print(f'\nAll plots saved.')

  Saved → /Users/mczhang/Documents/GitHub/FM7/02-data/email_plots/step06_confidence_hist.png


  Saved → /Users/mczhang/Documents/GitHub/FM7/02-data/email_plots/step06_polarity_summary.png


  Saved → /Users/mczhang/Documents/GitHub/FM7/02-data/email_plots/step06_sample_waveforms.png

All plots saved.


In [8]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=== DL Polarity Summary ===')
print(f'Events processed : {N:,}')
print(f'Stations         : {len(STATIONS_Y2)}')
print(f'Total picks      : {total_picks:,}')
print(f'High conf ≥{DL_CONF_MIN}   : {high_conf:,}  ({high_conf/max(total_picks,1)*100:.1f}%)')
print(f'Output file      : {OUT_H5}  ({OUT_H5.stat().st_size/1e6:.1f} MB)')
print()
print(f"{'Station':>6}  {'Picks':>6}  {'Hi-conf':>8}  {'Up':>6}  {'Down':>6}  {'U/D ratio':>10}")
print('-'*55)
for sta in STATIONS_Y2:
    n_picks = int((pol_out[sta] != 0).sum())
    n_hi    = int(((pol_out[sta] != 0) & (conf_out[sta] >= DL_CONF_MIN)).sum())
    n_up    = int((pol_out[sta] == 1).sum())
    n_dn    = int((pol_out[sta] == -1).sum())
    ratio   = n_up / max(n_dn, 1)
    flag    = ' ⚠' if ratio > 3 or ratio < 0.33 else ''
    print(f'{sta:>6}  {n_picks:>6,}  {n_hi:>8,}  {n_up:>6,}  {n_dn:>6,}  {ratio:>10.2f}{flag}')

print('\nNotebook 06 complete.')

=== DL Polarity Summary ===
Events processed : 1,182
Stations         : 22
Total picks      : 21,758
High conf ≥0.6   : 20,363  (93.6%)
Output file      : /Users/mczhang/Documents/GitHub/FM7/02-data/E_Po/dl_polarity_Y2_ge18.h5  (0.2 MB)

Station   Picks   Hi-conf      Up    Down   U/D ratio
-------------------------------------------------------
   AS1   1,178     1,137   1,067     111        9.61 ⚠
   AS2   1,179     1,109     654     525        1.25
   CC1   1,181     1,150   1,081     100       10.81 ⚠
   EC1   1,181     1,148     686     495        1.39
   EC2   1,173     1,115     753     420        1.79
   EC3   1,040       996     685     355        1.93
   ID1   1,069     1,003     668     401        1.67
   01B     735       649     489     246        1.99
   02B   1,058       950     727     331        2.20
   03B     862       806     494     368        1.34
   04B     892       812     525     367        1.43
   05B     628       591     366     262        1.40
   06B   1,1